In [12]:
import pandas as pd
import re
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alvar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\alvar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alvar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [6]:
df = pd.read_csv('espn_rss_noticias.csv', encoding='utf-8')
df.head()

,seccion,titulo,descripcion,enlace,fecha
0,general,Takeaways from Thunder-Lakers Game 4,Here's what we've learned -- and what's next f...,https://www.espn.com/nba/story/_/id/48676289/2...,"Tue, 12 May 2026 02:05:06 EST"
1,general,LeBron: 'I don't know what the future holds',"LeBron James, whose record-setting 23rd season...",https://www.espn.com/nba/story/_/id/48747842/l...,"Tue, 12 May 2026 07:00:09 EST"
2,general,OKC escapes late L.A. charge to complete sweep,The Lakers refused to fold while facing elimin...,https://www.espn.com/nba/story/_/id/48747206/t...,"Tue, 12 May 2026 07:00:09 EST"
3,general,Bickerstaff calls FT disparity in loss 'unacce...,Pistons coach J.B. Bickerstaff called the gulf...,https://www.espn.com/nba/story/_/id/48747058/b...,"Tue, 12 May 2026 07:00:09 EST"
4,general,Mitchell's record-tying 2nd half evens East semis,Donovan Mitchell tied an NBA playoff record wi...,https://www.espn.com/nba/story/_/id/48746487/d...,"Tue, 12 May 2026 07:00:09 EST"


In [7]:
df.info()
df[['seccion', 'titulo', 'descripcion']].head(3)

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   seccion      101 non-null    str  
 1   titulo       101 non-null    str  
 2   descripcion  101 non-null    str  
 3   enlace       101 non-null    str  
 4   fecha        101 non-null    str  
dtypes: str(5)
memory usage: 4.1 KB


,seccion,titulo,descripcion
0,general,Takeaways from Thunder-Lakers Game 4,Here's what we've learned -- and what's next f...
1,general,LeBron: 'I don't know what the future holds',"LeBron James, whose record-setting 23rd season..."
2,general,OKC escapes late L.A. charge to complete sweep,The Lakers refused to fold while facing elimin...


In [8]:
# nulos se pone continue string vacio
df['descripcion'] = df['descripcion'].fillna('')

#se limpia el texto
df['desc_limpia'] = df['descripcion'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x.lower()))

df[['descripcion', 'desc_limpia']].head(2)

,descripcion,desc_limpia
0,Here's what we've learned -- and what's next f...,heres what weve learned and whats next for th...
1,"LeBron James, whose record-setting 23rd season...",lebron james whose recordsetting rd season end...


In [ ]:
#tokenizanmos
df['tokens'] = df['desc_limpia'].apply(lambda x: word_tokenize(x))
df[['desc_limpia', 'tokens']].head(2)

,desc_limpia,tokens
0,heres what weve learned and whats next for th...,"[heres, what, weve, learned, and, whats, next,..."
1,lebron james whose recordsetting rd season end...,"[lebron, james, whose, recordsetting, rd, seas..."


In [14]:
stop_words = set(stopwords.words('english'))  

df['tokens_sin_stop'] = df['tokens'].apply(
    lambda lista: [pal for pal in lista if pal not in stop_words and len(pal) > 2]
)

df[['tokens', 'tokens_sin_stop']].head(2)


,tokens,tokens_sin_stop
0,"[heres, what, weve, learned, and, whats, next,...","[heres, weve, learned, whats, next, cavspiston..."
1,"[lebron, james, whose, recordsetting, rd, seas...","[lebron, james, whose, recordsetting, season, ..."


In [ ]:
# Cont palabras por noticia
df['num_palabras'] = df['tokens_sin_stop'].apply(lambda lista: len(lista))

# Long del titulo
df['long_titulo'] = df['titulo'].apply(lambda x: len(str(x)))

df[['seccion', 'num_palabras', 'long_titulo']].describe()


,num_palabras,long_titulo
count,101.000000,101.000000
mean,11.930693,51.366337
std,4.079847,6.209223
min,5.000000,34.000000
25%,9.000000,47.000000
50%,11.000000,53.000000
75%,14.000000,53.000000
max,26.000000,82.000000


In [16]:
# Unimos tokens limpios
todas_palabras = df['tokens_sin_stop'].sum()

frecuencias = FreqDist(todas_palabras)
frecuencias.most_common(15)

[('game', 11),
 ('heres', 10),
 ('season', 10),
 ('monday', 9),
 ('nba', 8),
 ('first', 8),
 ('series', 7),
 ('team', 7),
 ('espn', 7),
 ('draft', 7),
 ('secondround', 6),
 ('away', 6),
 ('night', 6),
 ('players', 6),
 ('offseason', 6)]

In [17]:
df.to_csv('espn_rss_noticias_procesado.csv', index=False, encoding='utf-8')


En esta practica use NLTK para procesar noticias RSS de ESPN. Le hice limpieza al texto, lo separe en palabras (token) y quite palabras que no aportan mucho, como "the", "a" o "is" (lo que aprendi que se llama stopwords). Tambien use funciones lambda con apply de pandas para transformar columnas y crear cosas como el numero de palabras de cada noticia y el largo del titulo. Al final saque cuales eran las palabras que más aparecian en las noticias.
